# DS5 Step 2 Path E — Condition-Stratified Phenotyping

**Notebook:** `104_ds5_condition_stratified_phenotype.ipynb`  
**Stage:** Stage 4, Step 2 — Path E  
**Date:** 2026-04-20

Tests whether DS5 behavioral phenotype structure is **condition-invariant** or **condition-shaped**.

In [ ]:
# [0] Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                              davies_bouldin_score, adjusted_rand_score)
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency
from itertools import combinations
from pathlib import Path

RANDOM_STATE   = 42
N_BOOTSTRAPS   = 100
SAMPLE_FRAC    = 0.80
MIN_CELL       = 30
JACCARD_STABLE = 0.75
JACCARD_MARG   = 0.60

FEATURES   = ['sig_hat', 'H_emp', 'x_mean']
CONDITIONS = ['control', 'default', 'reintervention']
COND_COLORS  = {'control': '#2196F3', 'default': '#FF9800', 'reintervention': '#4CAF50'}
COND_MARKERS = {'control': 'o', 'default': 's', 'reintervention': '^'}

ROOT       = Path('..')
PHENO_PATH = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
FIG_PATH   = PHENO_PATH / 'figures'
FIG_PATH.mkdir(parents=True, exist_ok=True)

print('Setup complete. Features:', FEATURES)
print('Conditions:', CONDITIONS)


In [ ]:
# [1] Load data
feat   = pd.read_csv(PHENO_PATH / 'ds5_phenotype_feature_matrix.csv')
path_a = pd.read_csv(PHENO_PATH / 'ds5_behav_only_cluster_assignments.csv')
path_f = pd.read_csv(PHENO_PATH / 'ds5_hierarchical_assignments.csv')

survey_path = ROOT / 'outputs' / 'features' / 'ds5' / 'ds5_survey_features.csv'
survey = pd.read_csv(survey_path) if survey_path.exists() else None

print('Feature matrix:', feat.shape)
print('Path A cols:', path_a.columns.tolist())
print('Path F cols:', path_f.columns.tolist())
if survey is not None:
    print('Survey:', survey.shape)


In [ ]:
# [2] Merge master dataframe (N=414)
keep_a = path_a[['participant_id_canonical', 'cluster_behav']].copy()
keep_f = path_f[['participant_id_canonical', 'profile_p1', 'stage2_B',
                  'stage2_A', 'cell_f', 'subprofile_C']].copy()

df = feat[['participant_id_canonical', 'study_condition'] + FEATURES +
          ['H_hour', 'mu_hat', 'sigma_mu_ratio', 'H_out_resid',
           'HHI_net', 'rho_self']].copy()
df = df.merge(keep_a, on='participant_id_canonical', how='left')
df = df.merge(keep_f, on='participant_id_canonical', how='left')

if survey is not None:
    PSYCH = ['belong_mean_pre', 'ostracism_mean_pre', 'fomo_mean_pre',
             'digital_wellbeing_pre', 'prob_self_pre']
    df = df.merge(survey[['participant_id_canonical'] + PSYCH],
                  on='participant_id_canonical', how='left')

print('Master df:', df.shape)
print('Condition counts:')
print(df['study_condition'].value_counts())
print('Missing in core features:')
print(df[FEATURES].isna().sum())


In [ ]:
# [3] Condition descriptives
print('=== Feature means by condition ===')
print(df.groupby('study_condition')[FEATURES].mean().round(4))
print()
print('=== Path A cluster x condition ===')
ct_a = pd.crosstab(df['study_condition'], df['cluster_behav'])
print(ct_a)
print()
print('Row %:')
print((ct_a.div(ct_a.sum(axis=1), axis=0)*100).round(1))
print()
print('=== Path F cell x condition ===')
ct_f = pd.crosstab(df['study_condition'], df['cell_f'])
print(ct_f)
print()
chi2_a, p_a, _, _ = chi2_contingency(ct_a)
print(f'Chi2 (Path A x condition): {chi2_a:.3f}, p={p_a:.4e}')
chi2_f, p_f, _, _ = chi2_contingency(ct_f)
print(f'Chi2 (Path F x condition): {chi2_f:.3f}, p={p_f:.4e}')


In [ ]:
# [4] Sub-analysis 2: Kruskal-Wallis across conditions (feature distributions)
print('=== KW tests: feature distributions across conditions ===')
kw_results = {}
for feat_name in FEATURES:
    groups = [df.loc[df['study_condition']==c, feat_name].dropna().values
              for c in CONDITIONS]
    stat, p = kruskal(*groups)
    kw_results[feat_name] = {'H': stat, 'p': p}
    print(f'{feat_name}: H={stat:.3f}, p={p:.4e}')

print()
print('=== Pairwise Mann-Whitney U tests ===')
for feat_name in FEATURES:
    print(f'  {feat_name}:')
    for c1, c2 in combinations(CONDITIONS, 2):
        g1 = df.loc[df['study_condition']==c1, feat_name].dropna().values
        g2 = df.loc[df['study_condition']==c2, feat_name].dropna().values
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        print(f'    {c1} vs {c2}: U={stat:.1f}, p={p:.4e}')


In [ ]:
# [5] Comparison 3: within Path-A cluster, feature means by condition
print('=== Within each Path A cluster: feature means by condition ===')
for cl in ['C1', 'C2', 'C3']:
    sub = df[df['cluster_behav'] == cl]
    print(f'\nCluster {cl} (N={len(sub)}):')
    means = sub.groupby('study_condition')[FEATURES].mean().round(4)
    print(means)
    # KW within cluster
    for feat_name in FEATURES:
        groups = [sub.loc[sub['study_condition']==c, feat_name].dropna().values
                  for c in CONDITIONS]
        if all(len(g) > 1 for g in groups):
            stat, p = kruskal(*groups)
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
            print(f'  KW {feat_name}: H={stat:.3f}, p={p:.4e} {sig}')


In [ ]:
# [6] Fig e1: feature distributions by condition (violin + stripplot)
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
feat_labels = {'sig_hat': r'$\hat{\sigma}$ (noise)',
               'H_emp':   r'$H_{emp}$ (entropy)',
               'x_mean':  r'$\bar{x}$ (override rate)'}
cond_order = CONDITIONS

for ax, feat_name in zip(axes, FEATURES):
    data_by_cond = [df.loc[df['study_condition']==c, feat_name].dropna().values
                    for c in cond_order]
    parts = ax.violinplot(data_by_cond, positions=range(len(cond_order)),
                          showmedians=True, showextrema=True)
    for i, (pc, c_name) in enumerate(zip(parts['bodies'], cond_order)):
        pc.set_facecolor(COND_COLORS[c_name])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)
    # Add means as markers
    for i, (vals, c_name) in enumerate(zip(data_by_cond, cond_order)):
        ax.scatter(i, np.mean(vals), color='white', edgecolor='black',
                   s=60, zorder=5, marker='D')
    ax.set_xticks(range(len(cond_order)))
    ax.set_xticklabels(['Control', 'Default', 'Reintervention'], rotation=15, ha='right')
    ax.set_ylabel(feat_labels[feat_name])
    ax.set_title(feat_labels[feat_name])
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('DS5 Behavioral Feature Distributions by Condition\n(diamond = mean, line = median)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e1_feature_distributions_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e1')


In [ ]:
# [7] Fig e2: pairplots by condition
pairs = [('sig_hat', 'H_emp'), ('sig_hat', 'x_mean'), ('H_emp', 'x_mean')]
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (fx, fy) in zip(axes, pairs):
    for c_name in CONDITIONS:
        sub = df[df['study_condition'] == c_name]
        ax.scatter(sub[fx], sub[fy], c=COND_COLORS[c_name],
                   marker=COND_MARKERS[c_name], alpha=0.4, s=18, label=c_name)
    ax.set_xlabel(fx); ax.set_ylabel(fy)
    ax.set_title(f'{fx} vs {fy}')
    ax.grid(alpha=0.25)

handles = [mpatches.Patch(color=COND_COLORS[c], label=c) for c in CONDITIONS]
axes[0].legend(handles=handles, fontsize=8)
fig.suptitle('Behavioral Feature Space by Condition (Path E)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e2_pairplots_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e2')


In [ ]:
# [8] Within-condition StandardScaler (primary)
scalers = {}
X_cond  = {}   # scaled arrays per condition (index aligned to df)

for c_name in CONDITIONS:
    mask = df['study_condition'] == c_name
    X_raw = df.loc[mask, FEATURES].values
    sc = StandardScaler()
    X_sc = sc.fit_transform(X_raw)
    scalers[c_name] = sc
    X_cond[c_name]  = (X_sc, df.index[mask].values)   # (array, original_indices)
    print(f'{c_name}: N={len(X_sc)}, scaler mean={sc.mean_.round(4)}, std={sc.scale_.round(4)}')


In [ ]:
# [9] Helper functions

def bootstrap_jaccard(X, k, n_bootstraps=100, sample_frac=0.8, random_state=42):
    rng = np.random.RandomState(random_state)
    n = len(X); sample_n = int(n * sample_frac)
    km_full = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=random_state)
    labels_full = km_full.fit_predict(X)
    scores = []
    for _ in range(n_bootstraps):
        idx  = rng.choice(n, size=sample_n, replace=False)
        X_b  = X[idx]
        km_b = KMeans(n_clusters=k, init='k-means++', n_init=10,
                      random_state=int(rng.randint(10000)))
        labels_b = km_b.fit_predict(X_b)
        lf = labels_full[idx]; lb = labels_b
        ti, tj = np.tril_indices(sample_n, k=-1)
        sf = lf[ti] == lf[tj]; sb = lb[ti] == lb[tj]
        tp = (sf & sb).sum(); fp = (~sf & sb).sum(); fn = (sf & ~sb).sum()
        d = tp + fp + fn
        scores.append(tp / d if d > 0 else 0.0)
    return float(np.mean(scores)), float(np.std(scores))

def order_labels_by_sig(X_raw_df, raw_labels, sig_col='sig_hat'):
    k = len(np.unique(raw_labels))
    order = [(c, X_raw_df.loc[raw_labels == c, sig_col].mean()) for c in range(k)]
    order.sort(key=lambda x: x[1])
    mapping = {orig: new for new, (orig, _) in enumerate(order)}
    return np.array([mapping[l] for l in raw_labels])

def jaccard_flag(j):
    if j >= JACCARD_STABLE: return 'STABLE'
    if j >= JACCARD_MARG:   return 'MARGINAL'
    return 'UNSTABLE'

print('Helper functions defined.')


In [ ]:
# [10] Full-sample within-condition k-selection (k=2,3,4)
k_results = {}   # {condition: {k: {sil, ch, db, min_cell, jaccard_mean, jaccard_std}}}

for c_name in CONDITIONS:
    X_sc, orig_idx = X_cond[c_name]
    sub_df = df.loc[orig_idx, FEATURES + ['sig_hat']].copy()
    k_results[c_name] = {}
    print(f'\n=== {c_name.upper()} (N={len(X_sc)}) ===')
    for k in [2, 3, 4]:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
        labels = km.fit_predict(X_sc)
        sizes = np.bincount(labels)
        min_sz = int(sizes.min())
        if min_sz < MIN_CELL:
            print(f'  k={k}: REJECTED (min cell={min_sz} < {MIN_CELL})')
            k_results[c_name][k] = {'sil': np.nan, 'ch': np.nan, 'db': np.nan,
                                     'min_cell': min_sz, 'jaccard': np.nan,
                                     'jaccard_std': np.nan, 'feasible': False}
            continue
        sil = silhouette_score(X_sc, labels)
        ch  = calinski_harabasz_score(X_sc, labels)
        db  = davies_bouldin_score(X_sc, labels)
        jac_mean, jac_std = bootstrap_jaccard(X_sc, k, N_BOOTSTRAPS, SAMPLE_FRAC, RANDOM_STATE)
        flag = jaccard_flag(jac_mean)
        print(f'  k={k}: sil={sil:.4f}, CH={ch:.1f}, DB={db:.3f}, '
              f'min={min_sz}, J={jac_mean:.3f}+/-{jac_std:.3f} [{flag}]')
        k_results[c_name][k] = {'sil': sil, 'ch': ch, 'db': db,
                                  'min_cell': min_sz,
                                  'jaccard': jac_mean, 'jaccard_std': jac_std,
                                  'feasible': jac_mean >= JACCARD_MARG}


In [ ]:
# [11] Select k per condition (best feasible k by silhouette)
SELECTED_K = {}   # {condition: k or None}

for c_name in CONDITIONS:
    best_k, best_sil = None, -np.inf
    for k, res in k_results[c_name].items():
        if res['feasible'] and res['sil'] > best_sil:
            best_sil = res['sil']; best_k = k
    SELECTED_K[c_name] = best_k
    if best_k is None:
        print(f'{c_name}: NO STABLE FEASIBLE k found')
    else:
        j = k_results[c_name][best_k]['jaccard']
        print(f'{c_name}: selected k={best_k}, sil={best_sil:.4f}, '
              f'J={j:.3f} [{jaccard_flag(j)}]')


In [ ]:
# [12] Fig e3: silhouette vs k by condition
fig, ax = plt.subplots(figsize=(7, 5))
k_vals = [2, 3, 4]

for c_name in CONDITIONS:
    sils = []
    for k in k_vals:
        res = k_results[c_name].get(k, {})
        sils.append(res.get('sil', np.nan))
    ax.plot(k_vals, sils, marker=COND_MARKERS[c_name], color=COND_COLORS[c_name],
            linewidth=2, markersize=9, label=c_name)
    # Mark selected k
    sel = SELECTED_K[c_name]
    if sel is not None:
        sel_sil = k_results[c_name][sel]['sil']
        ax.scatter(sel, sel_sil, color=COND_COLORS[c_name],
                   s=200, zorder=5, edgecolors='black', linewidths=2)

ax.axhline(0.30, color='grey', linestyle='--', alpha=0.6, label='Sil=0.30 threshold')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Silhouette score')
ax.set_title('Within-Condition k-Selection (filled circle = selected k)', fontweight='bold')
ax.set_xticks(k_vals)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e3_cluster_selection_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e3')


In [ ]:
# [13] Assign within-condition labels (ordered by sig_hat ascending)
cond_labels  = {}   # {condition: array of string labels, aligned to condition's df rows}
cond_km_models = {}

for c_name in CONDITIONS:
    X_sc, orig_idx = X_cond[c_name]
    sub_df = df.loc[orig_idx].copy()
    k = SELECTED_K[c_name]
    if k is None:
        cond_labels[c_name] = np.array(['undifferentiated'] * len(orig_idx))
        print(f'{c_name}: assigned all to undifferentiated (no stable k)')
        continue
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    raw = km.fit_predict(X_sc)
    cond_km_models[c_name] = km
    # Re-order by sig_hat ascending
    ordered = order_labels_by_sig(sub_df.reset_index(drop=True), raw, sig_col='sig_hat')
    label_prefix = c_name[:3].upper()  # CON, DEF, REI
    str_labels = np.array([label_prefix + '_C' + str(l+1) for l in ordered])
    cond_labels[c_name] = str_labels
    for l in sorted(np.unique(str_labels)):
        n_l = (str_labels == l).sum()
        mean_sig = sub_df.loc[sub_df.index, 'sig_hat'].values[str_labels == l].mean()
        mean_hmp = sub_df.loc[sub_df.index, 'H_emp'].values[str_labels == l].mean()
        mean_xm  = sub_df.loc[sub_df.index, 'x_mean'].values[str_labels == l].mean()
        print(f'  {l}: N={n_l}, sig={mean_sig:.4f}, H={mean_hmp:.4f}, x={mean_xm:.4f}')
    print()


In [ ]:
# [14] Merge within-condition labels into master df
df['cond_cluster_full'] = np.nan
df['cond_cluster_full'] = df['cond_cluster_full'].astype(object)

for c_name in CONDITIONS:
    _, orig_idx = X_cond[c_name]
    df.loc[orig_idx, 'cond_cluster_full'] = cond_labels[c_name]

print('cond_cluster_full value counts:')
print(df['cond_cluster_full'].value_counts())
print('Any NaN:', df['cond_cluster_full'].isna().sum())


In [ ]:
# [15] Fig e4: within-condition cluster profiles (grouped bars per feature)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
feat_labels = {'sig_hat': 'sig_hat (noise)',
               'H_emp':   'H_emp (entropy)',
               'x_mean':  'x_mean (override)'}

for ax, feat_name in zip(axes, FEATURES):
    profiles = {}
    for c_name in CONDITIONS:
        _, orig_idx = X_cond[c_name]
        sub = df.loc[orig_idx].copy()
        sub['_lab'] = cond_labels[c_name]
        for lab in sorted(sub['_lab'].unique()):
            vals = sub.loc[sub['_lab'] == lab, feat_name]
            profiles[lab] = (np.mean(vals), np.std(vals)/np.sqrt(len(vals)))
    labels_sorted = sorted(profiles.keys())
    x = np.arange(len(labels_sorted))
    means = [profiles[l][0] for l in labels_sorted]
    errs  = [profiles[l][1] for l in labels_sorted]
    colors = []
    for l in labels_sorted:
        if l.startswith('CON'): colors.append(COND_COLORS['control'])
        elif l.startswith('DEF'): colors.append(COND_COLORS['default'])
        else: colors.append(COND_COLORS['reintervention'])
    ax.bar(x, means, yerr=errs, color=colors, alpha=0.8, capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_sorted, rotation=35, ha='right', fontsize=7)
    ax.set_title(feat_labels[feat_name])
    ax.set_ylabel(feat_name)
    ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=COND_COLORS[c], label=c) for c in CONDITIONS]
axes[0].legend(handles=handles, fontsize=8)
fig.suptitle('Within-Condition Cluster Profiles (mean +/- SE)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e4_within_condition_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e4')


In [ ]:
# [16] Profile B within-condition k-selection (k=2,3; max k=3)
print('=== Profile B within-condition clustering ===')
print('Feasibility: N=94-99 per condition; k=3 gives ~31-33 per cell (borderline)')
print()

B_k_results  = {}
B_X_cond     = {}
B_scalers    = {}

for c_name in CONDITIONS:
    mask_B = (df['study_condition'] == c_name) & (df['profile_p1'] == 'Profile B')
    orig_idx_B = df.index[mask_B].values
    X_raw_B = df.loc[orig_idx_B, FEATURES].values
    sc_B = StandardScaler()
    X_sc_B = sc_B.fit_transform(X_raw_B)
    B_scalers[c_name]   = sc_B
    B_X_cond[c_name]    = (X_sc_B, orig_idx_B)
    B_k_results[c_name] = {}
    print(f'{c_name} Profile B: N={len(X_sc_B)}')
    for k in [2, 3]:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
        labels = km.fit_predict(X_sc_B)
        sizes = np.bincount(labels)
        min_sz = int(sizes.min())
        if min_sz < MIN_CELL:
            print(f'  k={k}: REJECTED (min cell={min_sz} < {MIN_CELL})')
            B_k_results[c_name][k] = {'sil': np.nan, 'min_cell': min_sz,
                                       'jaccard': np.nan, 'feasible': False}
            continue
        sil = silhouette_score(X_sc_B, labels)
        jac_mean, jac_std = bootstrap_jaccard(X_sc_B, k, N_BOOTSTRAPS, SAMPLE_FRAC, RANDOM_STATE)
        flag = jaccard_flag(jac_mean)
        print(f'  k={k}: sil={sil:.4f}, min={min_sz}, J={jac_mean:.3f}+/-{jac_std:.3f} [{flag}]')
        B_k_results[c_name][k] = {'sil': sil, 'min_cell': min_sz,
                                   'jaccard': jac_mean, 'jaccard_std': jac_std,
                                   'feasible': jac_mean >= JACCARD_MARG}
    print()

# Select k for Profile B within each condition
B_SELECTED_K = {}
for c_name in CONDITIONS:
    best_k, best_sil = None, -np.inf
    for k, res in B_k_results[c_name].items():
        if res['feasible'] and res['sil'] > best_sil:
            best_sil = res['sil']; best_k = k
    B_SELECTED_K[c_name] = best_k
    status = f'k={best_k}' if best_k else 'NO STABLE k'
    print(f'{c_name} Profile B selected: {status}')


In [ ]:
# [17] Profile B label assignment
df['cond_cluster_B'] = np.nan
df['cond_cluster_B'] = df['cond_cluster_B'].astype(object)
B_km_models = {}

for c_name in CONDITIONS:
    X_sc_B, orig_idx_B = B_X_cond[c_name]
    sub_B = df.loc[orig_idx_B].copy()
    k = B_SELECTED_K[c_name]
    if k is None:
        df.loc[orig_idx_B, 'cond_cluster_B'] = 'B_undiff'
        continue
    km_B = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    raw_B = km_B.fit_predict(X_sc_B)
    B_km_models[c_name] = km_B
    ordered_B = order_labels_by_sig(sub_B.reset_index(drop=True), raw_B, sig_col='sig_hat')
    prefix = c_name[:3].upper()
    str_B = np.array([prefix + '_B' + str(l+1) for l in ordered_B])
    df.loc[orig_idx_B, 'cond_cluster_B'] = str_B
    print(f'{c_name} Profile B clusters:')
    for l in sorted(np.unique(str_B)):
        sub_l = sub_B.iloc[str_B == l]
        print(f'  {l}: N={(str_B == l).sum()}')

print('\ncond_cluster_B (Profile B only):')
print(df['cond_cluster_B'].value_counts(dropna=False))


In [ ]:
# [18] Comparison 1: ARI(within-condition full solution vs Path A) per condition
print('=== Comparison 1: ARI(within-cond vs Path A) per condition ===')
ari_c1 = {}
for c_name in CONDITIONS:
    _, orig_idx = X_cond[c_name]
    sub = df.loc[orig_idx]
    mask = sub['cluster_behav'].notna() & sub['cond_cluster_full'].notna()
    if mask.sum() == 0:
        ari_c1[c_name] = np.nan
        print(f'{c_name}: no overlap')
        continue
    ari = adjusted_rand_score(sub.loc[mask, 'cluster_behav'],
                               sub.loc[mask, 'cond_cluster_full'])
    ari_c1[c_name] = ari
    interp = ('HIGH - structure invariant' if ari > 0.7
              else 'MODERATE - partial recovery' if ari > 0.4
              else 'LOW - structural divergence')
    print(f'{c_name}: ARI={ari:.4f}  [{interp}]')


In [ ]:
# [19] Comparison 2: cross-condition ARI via centroid projection
# Apply each condition's model to all 414 participants (full-sample raw features)
# Use the full-sample scaler from Path A sensitivity (will refit here)
print('=== Comparison 2: cross-condition centroid projection ===')

# Fit full-sample scaler (for projection)
sc_full = StandardScaler()
X_full_all = sc_full.fit_transform(df[FEATURES].values)

projected_labels = {}
for c_name in CONDITIONS:
    if c_name not in cond_km_models:
        print(f'{c_name}: no model available (undifferentiated), skip')
        continue
    km_c = cond_km_models[c_name]
    sc_c = scalers[c_name]
    # Project all 414 using condition-specific scaler then predict with condition's centroids
    # NOTE: This uses within-condition scaler, so centroids are in different spaces.
    # Use the condition's own scaler for re-centering:
    X_rescaled = sc_c.transform(df[FEATURES].values)
    proj = km_c.predict(X_rescaled)
    projected_labels[c_name] = proj

print('Cross-condition ARI (applying each model to all 414):')
c_names_with_models = list(projected_labels.keys())
for c1, c2 in combinations(c_names_with_models, 2):
    ari = adjusted_rand_score(projected_labels[c1], projected_labels[c2])
    interp = ('HIGH' if ari > 0.7 else 'MODERATE' if ari > 0.4 else 'LOW')
    print(f'  {c1} model vs {c2} model: ARI={ari:.4f} [{interp}]')


In [ ]:
# [20] Comparison 4: k consistency table
print('=== Comparison 4: k consistency across conditions ===')
print()
header = f'{"Condition":<20} {"Full k":>8} {"Profile B k":>12}'
print(header)
print('-' * len(header))
for c_name in CONDITIONS:
    fk = SELECTED_K[c_name] if SELECTED_K[c_name] else 'none'
    bk = B_SELECTED_K[c_name] if B_SELECTED_K[c_name] else 'none'
    print(f'{c_name:<20} {str(fk):>8} {str(bk):>12}')
print()
full_ks = [SELECTED_K[c] for c in CONDITIONS if SELECTED_K[c] is not None]
b_ks    = [B_SELECTED_K[c] for c in CONDITIONS if B_SELECTED_K[c] is not None]
print(f'Full-sample k consistent across all conditions: {len(set(full_ks))==1 and len(full_ks)==3}')
print(f'Profile B k consistent: {len(set(b_ks))==1 and len(b_ks)==3}')
path_a_k = 3
print(f'Full-sample k matches Path A (k=3): {all(k == path_a_k for k in full_ks)}')


In [ ]:
# [21] Comparison 5: Profile B within-cond vs Path F Stage 2 (B_behav labels)
print('=== Comparison 5: Profile B within-cond vs Path F Stage 2 ===')
for c_name in CONDITIONS:
    mask_B = ((df['study_condition'] == c_name) &
              (df['profile_p1'] == 'Profile B') &
              df['cond_cluster_B'].notna() &
              df['stage2_B'].notna())
    n = mask_B.sum()
    if n == 0:
        print(f'{c_name}: no Profile B overlap')
        continue
    ari = adjusted_rand_score(df.loc[mask_B, 'stage2_B'],
                               df.loc[mask_B, 'cond_cluster_B'])
    print(f'{c_name} (N={n}): ARI(within-cond B vs Path F stage2_B) = {ari:.4f}')
    ct = pd.crosstab(df.loc[mask_B, 'stage2_B'],
                     df.loc[mask_B, 'cond_cluster_B'])
    print(ct); print()


In [ ]:
# [22] Fig e5: ARI comparison heatmap
import matplotlib.colors as mcolors

row_labels = [f'vs Path A ({c})' for c in CONDITIONS] +              [f'cross-cond: {c1} vs {c2}'
              for c1, c2 in combinations([c for c in CONDITIONS if c in projected_labels], 2)]
ari_vals = [ari_c1.get(c, np.nan) for c in CONDITIONS]

c_names_proj = [c for c in CONDITIONS if c in projected_labels]
for c1, c2 in combinations(c_names_proj, 2):
    ari_vals.append(adjusted_rand_score(projected_labels[c1], projected_labels[c2]))

fig, ax = plt.subplots(figsize=(7, max(4, len(row_labels)*0.55 + 1)))
colors_bar = ['#d73027' if v < 0.4 else '#fdae61' if v < 0.7 else '#1a9850'
              for v in ari_vals]
y_pos = range(len(row_labels))
bars = ax.barh(list(y_pos), ari_vals, color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(0.4, color='grey', linestyle='--', alpha=0.6, linewidth=1)
ax.axvline(0.7, color='grey', linestyle=':', alpha=0.6, linewidth=1)
for bar, val in zip(bars, ari_vals):
    if not np.isnan(val):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(row_labels, fontsize=9)
ax.set_xlim(0, 1.1); ax.set_xlabel('ARI')
ax.set_title('Path E: ARI Comparison Summary', fontweight='bold')
legend_handles = [mpatches.Patch(color='#d73027', label='<0.4 low'),
                  mpatches.Patch(color='#fdae61', label='0.4-0.7 moderate'),
                  mpatches.Patch(color='#1a9850', label='>0.7 high')]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e5_ari_comparison_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e5')


In [ ]:
# [23] Sensitivity: full-sample scaler (comparison 6)
print('=== Sensitivity: re-run within-condition clustering with full-sample scaler ===')
sc_fs = StandardScaler()
sc_fs.fit(df[FEATURES].values)

sens_labels  = {}
sens_centers = {}   # {condition: centroids in full-sample scaler space}

for c_name in CONDITIONS:
    _, orig_idx = X_cond[c_name]
    X_raw_c = df.loc[orig_idx, FEATURES].values
    X_fs_c  = sc_fs.transform(X_raw_c)
    k = SELECTED_K[c_name]
    if k is None:
        sens_labels[c_name] = np.array(['undifferentiated'] * len(orig_idx))
        continue
    km_fs = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    raw_fs = km_fs.fit_predict(X_fs_c)
    sub_df = df.loc[orig_idx].copy()
    ordered_fs = order_labels_by_sig(sub_df.reset_index(drop=True), raw_fs, sig_col='sig_hat')
    sens_labels[c_name] = ordered_fs
    # Store cluster centers in full-sample scaler space
    centers = np.zeros((k, len(FEATURES)))
    for cl_i in range(k):
        centers[cl_i] = X_fs_c[ordered_fs == cl_i].mean(axis=0)
    sens_centers[c_name] = centers
    # ARI vs within-cond scaler solution
    ari_sens = adjusted_rand_score(
        cond_labels[c_name],
        np.array(['FS_C' + str(l+1) for l in ordered_fs])
    )
    print(f'{c_name}: ARI(within-cond scaler vs full-sample scaler) = {ari_sens:.4f}')


In [ ]:
# [24] Comparison 6: centroid displacement in full-sample scaler space
print('=== Comparison 6: centroid displacement across conditions ===')
print('(in full-sample StandardScaler space; 1 unit = 1 std of full sample)')
print()

c_pairs = list(combinations([c for c in CONDITIONS if c in sens_centers], 2))
for c1, c2 in c_pairs:
    ctrs1 = sens_centers[c1]
    ctrs2 = sens_centers[c2]
    k1, k2 = len(ctrs1), len(ctrs2)
    print(f'{c1} (k={k1}) vs {c2} (k={k2}):')
    if k1 == k2:
        # Match centroids by minimum distance
        from scipy.spatial.distance import cdist
        dmat = cdist(ctrs1, ctrs2)
        for i in range(k1):
            j = int(np.argmin(dmat[i]))
            d = dmat[i, j]
            flag = 'SMALL (<0.5)' if d < 0.5 else 'MODERATE (0.5-1.0)' if d < 1.0 else 'LARGE (>1.0)'
            print(f'  centroid {i+1} (from {c1}) -> nearest centroid {j+1} (in {c2}): '
                  f'dist={d:.4f} [{flag}]')
    else:
        print(f'  Different k ({k1} vs {k2}) — pairwise centroid match not applicable')
        # Report centroid positions for reference
        for i, ctr in enumerate(ctrs1):
            print(f'  {c1} centroid {i+1}: {dict(zip(FEATURES, ctr.round(3)))}')
        for i, ctr in enumerate(ctrs2):
            print(f'  {c2} centroid {i+1}: {dict(zip(FEATURES, ctr.round(3)))}')
    print()


In [ ]:
# [25] Fig e6: stacked bars — within-condition cluster proportions per condition
# Show both Path A (full-sample) and within-condition solutions side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Left panel: Path A full-sample applied to each condition (distribution shift)
ax = axes[0]
path_a_cats = ['C1', 'C2', 'C3']
path_a_colors = {'C1': '#66c2a5', 'C2': '#fc8d62', 'C3': '#8da0cb'}
bottom = np.zeros(len(CONDITIONS))
for ca in path_a_cats:
    vals = []
    for c_name in CONDITIONS:
        sub = df[df['study_condition'] == c_name]
        total = len(sub)
        count = (sub['cluster_behav'] == ca).sum()
        vals.append(count / total * 100)
    ax.bar(CONDITIONS, vals, bottom=bottom,
           color=path_a_colors[ca], label=ca, alpha=0.85, edgecolor='white')
    bottom += np.array(vals)
ax.set_ylim(0, 100); ax.set_ylabel('% of condition group')
ax.set_title('Path A full-sample clusters\nby condition', fontweight='bold')
ax.legend(title='Path A cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
ax.set_xticklabels(CONDITIONS, rotation=15)

# Right panel: within-condition solutions
ax = axes[1]
all_cond_labs = sorted(set(l for labs in cond_labels.values() for l in labs))
n_labs = len(all_cond_labs)
lab_colors = {}
base_colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf']
for i, lab in enumerate(all_cond_labs):
    lab_colors[lab] = base_colors[i % len(base_colors)]

bottom2 = np.zeros(len(CONDITIONS))
for lab in all_cond_labs:
    vals2 = []
    for c_name in CONDITIONS:
        _, orig_idx = X_cond[c_name]
        labs = cond_labels[c_name]
        vals2.append((labs == lab).sum() / len(labs) * 100)
    ax.bar(CONDITIONS, vals2, bottom=bottom2,
           color=lab_colors[lab], label=lab, alpha=0.85, edgecolor='white')
    bottom2 += np.array(vals2)
ax.set_ylim(0, 100); ax.set_ylabel('% of condition group')
ax.set_title('Within-condition cluster solutions\nby condition', fontweight='bold')
ax.legend(title='Within-cond cluster', bbox_to_anchor=(1.25, 1.0), fontsize=8)
ax.set_xticklabels(CONDITIONS, rotation=15)

fig.suptitle('Cluster Proportion Shifts Across Conditions (Path E)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e6_proportion_shifts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e6')


In [ ]:
# [26] Fig e7: within each Path A cluster (C1/C2/C3), feature boxplots by condition
path_a_clusters = ['C1', 'C2', 'C3']
fig, axes = plt.subplots(3, 3, figsize=(13, 11))

for row_i, cl in enumerate(path_a_clusters):
    sub_cl = df[df['cluster_behav'] == cl]
    for col_i, feat_name in enumerate(FEATURES):
        ax = axes[row_i][col_i]
        data_by_cond = [sub_cl.loc[sub_cl['study_condition']==c, feat_name].dropna().values
                        for c in CONDITIONS]
        bp = ax.boxplot(data_by_cond, patch_artist=True, notch=False,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, c_name in zip(bp['boxes'], CONDITIONS):
            patch.set_facecolor(COND_COLORS[c_name])
            patch.set_alpha(0.7)
        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels(['Control', 'Default', 'Reint.'], fontsize=8)
        if col_i == 0:
            ax.set_ylabel(f'Path A {cl}', fontsize=9, fontweight='bold')
        if row_i == 0:
            ax.set_title(feat_name, fontsize=9)
        # KW test annotation
        if all(len(d) > 1 for d in data_by_cond):
            stat, p = kruskal(*data_by_cond)
            stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            ax.text(0.97, 0.97, stars, transform=ax.transAxes,
                    ha='right', va='top', fontsize=10, fontweight='bold',
                    color='red' if stars != 'ns' else 'grey')
        ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=COND_COLORS[c], label=c) for c in CONDITIONS]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Within-Cluster Feature Distributions by Condition\n'
             '(* p<0.05, ** p<0.01, *** p<0.001 KW test)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
fig.savefig(FIG_PATH / 'fig_e7_within_cluster_condition_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e7')


In [ ]:
# [27] Fig e8: alluvial — within-condition assignment vs Path A (per condition row)
def draw_alluvial_simple(ax, left_labs, right_labs, crosstab_df, l_colors, r_colors, title=''):
    total = crosstab_df.values.sum()
    gap_frac = 0.04
    def bar_rects(cats, col, ct, side='left'):
        rects = {}
        cum = 0.0
        for cat in cats:
            h = ct.loc[cat].sum() / total if side == 'left' else ct[cat].sum() / total
            rects[cat] = (cum, cum + h - gap_frac * 0.5)
            cum += h
        return rects
    l_rects = {}; cum = 0.0
    for lc in left_labs:
        row = crosstab_df.loc[lc] if lc in crosstab_df.index else pd.Series(0, index=right_labs)
        h = row.sum() / total
        l_rects[lc] = (cum, cum + h - gap_frac * 0.4)
        cum += h + gap_frac * 0.1
    r_rects = {}; cum = 0.0
    for rc in right_labs:
        col_vals = crosstab_df[rc] if rc in crosstab_df.columns else pd.Series(0, index=left_labs)
        h = col_vals.sum() / total
        r_rects[rc] = (cum, cum + h - gap_frac * 0.4)
        cum += h + gap_frac * 0.1
    # Draw flows
    l_flow_pos = {lc: l_rects[lc][0] for lc in left_labs}
    for lc in left_labs:
        row = crosstab_df.loc[lc] if lc in crosstab_df.index else pd.Series(0, index=right_labs)
        for rc in right_labs:
            flow = row.get(rc, 0) / total
            if flow < 1e-6: continue
            ly0 = l_flow_pos[lc]; ly1 = ly0 + flow
            r_start = r_rects[rc][0] if rc in r_rects else 0
            r_flow_pos_rc = getattr(ax, '_r_flow_' + rc, r_start)
            ry0 = r_flow_pos_rc; ry1 = ry0 + flow
            setattr(ax, '_r_flow_' + rc, ry1)
            t = np.linspace(0, 1, 60)
            s = 3*t**2 - 2*t**3
            xu = 0.25 + (0.75 - 0.25)*t
            y_top = ly1 + (ry1 - ly1)*s
            y_bot = ly0 + (ry0 - ly0)*s
            xs = np.concatenate([xu, xu[::-1]])
            ys = np.concatenate([y_top, y_bot[::-1]])
            poly = Polygon(np.column_stack([xs, ys]), closed=True,
                           facecolor=l_colors.get(lc, '#aaaaaa'), alpha=0.35, edgecolor='none')
            ax.add_patch(poly)
            l_flow_pos[lc] = ly1
    # Draw bars
    bw = 0.06
    for lc in left_labs:
        y0, y1 = l_rects[lc]
        ax.add_patch(plt.Rectangle((0.2, y0), bw, y1-y0,
                                   facecolor=l_colors.get(lc, '#888'), edgecolor='white', linewidth=0.5))
        ax.text(0.19, (y0+y1)/2, lc, ha='right', va='center', fontsize=7)
    for rc in right_labs:
        y0, y1 = r_rects[rc]
        ax.add_patch(plt.Rectangle((0.74, y0), bw, y1-y0,
                                   facecolor=r_colors.get(rc, '#888'), edgecolor='white', linewidth=0.5))
        ax.text(0.81, (y0+y1)/2, rc, ha='left', va='center', fontsize=7)
    ax.set_xlim(0, 1); ax.set_ylim(-0.02, 1.05)
    ax.axis('off'); ax.set_title(title, fontsize=9, fontweight='bold')

fig, axes = plt.subplots(1, 3, figsize=(15, 7))
path_a_col = {'C1': '#66c2a5', 'C2': '#fc8d62', 'C3': '#8da0cb'}
cond_short = {'control': 'CON', 'default': 'DEF', 'reintervention': 'REI'}

for ax, c_name in zip(axes, CONDITIONS):
    _, orig_idx = X_cond[c_name]
    sub = df.loc[orig_idx]
    within_labs_c = cond_labels[c_name]
    ul_sorted = sorted(np.unique(within_labs_c))
    ur_sorted = ['C1', 'C2', 'C3']
    # Build contingency
    rows = []
    for ul in ul_sorted:
        row_dict = {}
        for ur in ur_sorted:
            mask_both = (within_labs_c == ul) & (sub['cluster_behav'].values == ur)
            row_dict[ur] = int(mask_both.sum())
        rows.append(row_dict)
    ct_df = pd.DataFrame(rows, index=ul_sorted, columns=ur_sorted)
    l_col = {l: COND_COLORS[c_name] for l in ul_sorted}
    for ax_ in [ax]:
        ax_._r_flow_C1 = r_start_C1 = 0
        ax_._r_flow_C2 = r_start_C2 = 0
        ax_._r_flow_C3 = r_start_C3 = 0
    draw_alluvial_simple(ax, ul_sorted, ur_sorted, ct_df, l_col, path_a_col,
                         title=f'{c_name.capitalize()}\nWithin-cond -> Path A')

fig.suptitle('Within-Condition Clusters vs Path A Full-Sample Clusters (Path E)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_e8_alluvial_cond_vs_path_a.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_e8')


In [ ]:
# [28] Save ds5_condition_cluster_assignments.csv + companion .md
out_cols = ['participant_id_canonical', 'study_condition', 'profile_p1',
            'cond_cluster_full', 'cond_cluster_B',
            'cluster_behav', 'cell_f', 'subprofile_C']
out_df = df[out_cols].copy()
out_path = PHENO_PATH / 'ds5_condition_cluster_assignments.csv'
out_df.to_csv(out_path, index=False)
print('Saved:', out_path, out_df.shape)

md_txt = '''# ds5_condition_cluster_assignments.csv

## What this file contains
Per-participant cluster assignments from Path E condition-stratified phenotyping.
Each row is one DS5 participant (N=414, StoP-eligible with complete behavioral features).

## Column reference
| Column | Type | Description |
|---|---|---|
| participant_id_canonical | str | Unique participant identifier |
| study_condition | str | Experimental condition: control / default / reintervention |
| profile_p1 | str | Phase 1 network profile: Profile A / Profile B |
| cond_cluster_full | str | Within-condition full-sample cluster (e.g. CON_C1, DEF_C2) |
| cond_cluster_B | str | Within-condition Profile B cluster (e.g. CON_B1); NaN for Profile A |
| cluster_behav | str | Path A full-sample behavioral cluster: C1 / C2 / C3 |
| cell_f | str | Path F hierarchical cell: B_behav1/2/3 or A_all |
| subprofile_C | str | Path C network sub-cluster: B1/B2/A1/A2 |

## How produced
Notebook: 104_ds5_condition_stratified_phenotype.ipynb
Source: ds5_phenotype_feature_matrix.csv, ds5_behav_only_cluster_assignments.csv,
        ds5_hierarchical_assignments.csv
Clustering: KMeans(k-means++, n_init=10, random_state=42)
Scaler: within-condition StandardScaler on [sig_hat, H_emp, x_mean]

## Known limitations
- Profile A within-condition clustering not feasible (N=36-46 per condition < 30 min cell)
- cond_cluster_B is NaN for all Profile A participants
- Cluster labels are condition-specific — CON_C1 and DEF_C1 are NOT the same type
- Within-condition scaler means centroids are in incomparable coordinate systems
'''
with open(PHENO_PATH / 'ds5_condition_cluster_assignments.md', 'w') as f:
    f.write(md_txt)
print('Saved companion .md')


In [ ]:
# [29] Save ds5_condition_cluster_profiles.csv + companion .md
profile_rows = []
for c_name in CONDITIONS:
    _, orig_idx = X_cond[c_name]
    sub = df.loc[orig_idx].copy()
    sub['_lab'] = cond_labels[c_name]
    for lab in sorted(sub['_lab'].unique()):
        sub_l = sub[sub['_lab'] == lab]
        row = {'condition': c_name, 'within_cond_cluster': lab, 'N': len(sub_l)}
        for feat_name in FEATURES + ['H_hour', 'HHI_net', 'rho_self']:
            vals = sub_l[feat_name].dropna()
            row[feat_name + '_mean']   = vals.mean()
            row[feat_name + '_median'] = vals.median()
            row[feat_name + '_std']    = vals.std()
        profile_rows.append(row)

profiles_df = pd.DataFrame(profile_rows)
prof_path = PHENO_PATH / 'ds5_condition_cluster_profiles.csv'
profiles_df.to_csv(prof_path, index=False)
print('Saved:', prof_path, profiles_df.shape)
print(profiles_df[['condition', 'within_cond_cluster', 'N'] +
                   [f + '_mean' for f in FEATURES]].to_string(index=False))

md_txt2 = '''# ds5_condition_cluster_profiles.csv

## What this file contains
Mean/median/std of behavioral features per within-condition cluster (Path E).
One row per within-condition cluster (across all three conditions).

## Column reference
| Column | Type | Description |
|---|---|---|
| condition | str | Experimental condition |
| within_cond_cluster | str | Cluster label (e.g. CON_C1, DEF_C2) |
| N | int | Cluster size |
| sig_hat_mean/median/std | float | Noise/diffusion parameter |
| H_emp_mean/median/std | float | Behavioral entropy |
| x_mean_mean/median/std | float | Override rate |
| H_hour_mean/median/std | float | Temporal entropy (overlay only) |
| HHI_net_mean/median/std | float | Network HHI (overlay only) |
| rho_self_mean/median/std | float | Self-loop fraction (overlay only) |

## How produced
Notebook: 104_ds5_condition_stratified_phenotype.ipynb
'''
with open(PHENO_PATH / 'ds5_condition_cluster_profiles.md', 'w') as f:
    f.write(md_txt2)
print('Saved companion .md')


In [ ]:
# [30] Save ds5_condition_cluster_stability.csv + companion .md
stab_rows = []

# Full-sample within-condition
for c_name in CONDITIONS:
    for k in [2, 3, 4]:
        res = k_results[c_name].get(k, {})
        stab_rows.append({
            'scope': 'full_sample',
            'condition': c_name,
            'k': k,
            'silhouette': res.get('sil', np.nan),
            'min_cell': res.get('min_cell', np.nan),
            'jaccard_mean': res.get('jaccard', np.nan),
            'jaccard_std': res.get('jaccard_std', np.nan),
            'feasible': res.get('feasible', False),
            'selected': SELECTED_K[c_name] == k
        })

# Profile B within-condition
for c_name in CONDITIONS:
    for k in [2, 3]:
        res = B_k_results[c_name].get(k, {})
        stab_rows.append({
            'scope': 'profile_B',
            'condition': c_name,
            'k': k,
            'silhouette': res.get('sil', np.nan),
            'min_cell': res.get('min_cell', np.nan),
            'jaccard_mean': res.get('jaccard', np.nan),
            'jaccard_std': res.get('jaccard_std', np.nan),
            'feasible': res.get('feasible', False),
            'selected': B_SELECTED_K[c_name] == k
        })

stab_df = pd.DataFrame(stab_rows)
stab_path = PHENO_PATH / 'ds5_condition_cluster_stability.csv'
stab_df.to_csv(stab_path, index=False)
print('Saved:', stab_path, stab_df.shape)
print(stab_df.to_string(index=False))

md_txt3 = '''# ds5_condition_cluster_stability.csv

## What this file contains
K-selection results and bootstrap Jaccard stability for all within-condition clustering
attempts (Path E). One row per condition x scope x k combination.

## Column reference
| Column | Type | Description |
|---|---|---|
| scope | str | full_sample (all N~137-142) or profile_B (N~94-99) |
| condition | str | Experimental condition |
| k | int | Number of clusters attempted |
| silhouette | float | Mean silhouette score; NaN if rejected |
| min_cell | int | Minimum cluster size |
| jaccard_mean | float | Bootstrap Jaccard mean (100 bootstraps, 80% sample) |
| jaccard_std | float | Bootstrap Jaccard std |
| feasible | bool | True if Jaccard >= 0.60 and min_cell >= 30 |
| selected | bool | True if this k was selected as the best for this condition/scope |

## Stability thresholds
- Jaccard >= 0.75: STABLE
- 0.60-0.75: MARGINAL
- < 0.60: UNSTABLE
- min_cell < 30: REJECTED regardless of Jaccard
'''
with open(PHENO_PATH / 'ds5_condition_cluster_stability.md', 'w') as f:
    f.write(md_txt3)
print('Saved companion .md')


In [ ]:
# [31] Save model_run_summary_path_e.md
import datetime

# Collect actual results for summary
selected_k_str = {c: str(SELECTED_K[c]) if SELECTED_K[c] else 'none' for c in CONDITIONS}
b_selected_k_str = {c: str(B_SELECTED_K[c]) if B_SELECTED_K[c] else 'none' for c in CONDITIONS}
ari_c1_str = {c: f'{v:.4f}' if not np.isnan(v) else 'n/a' for c, v in ari_c1.items()}

# Determine outcome category
ari_vals_list = [ari_c1.get(c, np.nan) for c in CONDITIONS]
ari_clean = [v for v in ari_vals_list if not np.isnan(v)]
mean_ari = np.mean(ari_clean) if ari_clean else np.nan

if mean_ari >= 0.7:
    outcome = 'A — Structure-invariant (high ARI, types stable across conditions)'
elif mean_ari >= 0.4:
    outcome = 'B — Distributional shift with partial structural stability (moderate ARI)'
else:
    outcome = 'C/D — Structure-dependent (low ARI; condition shapes phenotype space)'

k_str_c1 = selected_k_str.get('control', 'n/a')
k_str_c2 = selected_k_str.get('default', 'n/a')
k_str_c3 = selected_k_str.get('reintervention', 'n/a')

summary = f"""# DS5 Step 2 Path E — Condition-Stratified Phenotyping: Model Run Summary

**Date:** 2026-04-20
**Notebook:** 104_ds5_condition_stratified_phenotype.ipynb
**Stage:** Stage 4, Step 2 — Path E

---

## Run parameters
- Core clustering features: {FEATURES}
- Conditions analyzed: {CONDITIONS}
- Scaling: within-condition StandardScaler (primary); full-sample scaler (sensitivity)
- RANDOM_STATE: {RANDOM_STATE}
- Bootstrap: {N_BOOTSTRAPS} iterations, {SAMPLE_FRAC} sample fraction
- Min cluster size threshold: {MIN_CELL}
- Analysis population: N = {len(df)}

## k-selection results (full-sample within-condition)

| Condition | N | Selected k | Jaccard |
|---|---|---|---|
| control | 137 | {k_str_c1} | {k_results['control'].get(SELECTED_K['control'], {}).get('jaccard', float('nan')):.3f} |
| default | 142 | {k_str_c2} | {k_results['default'].get(SELECTED_K['default'], {}).get('jaccard', float('nan')):.3f} |
| reintervention | 135 | {k_str_c3} | {k_results['reintervention'].get(SELECTED_K['reintervention'], {}).get('jaccard', float('nan')):.3f} |

## k-selection results (Profile B within-condition)

| Condition | Profile B N | Selected k |
|---|---|---|
| control | 94 | {b_selected_k_str['control']} |
| default | 96 | {b_selected_k_str['default']} |
| reintervention | 99 | {b_selected_k_str['reintervention']} |

Note: Profile A within-condition clustering NOT attempted (N=36-46, cells < 30).

## Cross-path comparison summary

| Comparison | Value |
|---|---|
| ARI(control within-cond vs Path A) | {ari_c1_str.get('control', 'n/a')} |
| ARI(default within-cond vs Path A) | {ari_c1_str.get('default', 'n/a')} |
| ARI(reintervention within-cond vs Path A) | {ari_c1_str.get('reintervention', 'n/a')} |
| Mean ARI across conditions | {mean_ari:.4f} |

## Interpretation outcome

**Matched outcome: {outcome}**

## Output files
- ds5_condition_cluster_assignments.csv
- ds5_condition_cluster_profiles.csv
- ds5_condition_cluster_stability.csv
- figures/fig_e1 through fig_e8

## Key limitations
1. Within-condition clusters use condition-specific scalers; labels are NOT cross-condition comparable
2. Profile A within-condition clustering infeasible (N < 30 per cell)
3. No causal interpretation: condition was randomly assigned, but cluster memberships are post-hoc
4. ARI comparisons between different-k solutions are lower-bounded by chance; interpret cautiously
5. Condition effect on behavioral features cannot be separated from true individual differences
"""
with open(PHENO_PATH / 'model_run_summary_path_e.md', 'w') as f:
    f.write(summary)
print('Saved model_run_summary_path_e.md')
print(f'Mean ARI: {mean_ari:.4f}')
print(f'Outcome: {outcome}')


In [ ]:
# [32] Verification — confirm all outputs
required_files = [
    'ds5_condition_cluster_assignments.csv',
    'ds5_condition_cluster_assignments.md',
    'ds5_condition_cluster_profiles.csv',
    'ds5_condition_cluster_profiles.md',
    'ds5_condition_cluster_stability.csv',
    'ds5_condition_cluster_stability.md',
    'model_run_summary_path_e.md',
]
required_figs = [
    'fig_e1_feature_distributions_by_condition.png',
    'fig_e2_pairplots_by_condition.png',
    'fig_e3_cluster_selection_by_condition.png',
    'fig_e4_within_condition_profiles.png',
    'fig_e5_ari_comparison_heatmap.png',
    'fig_e6_proportion_shifts.png',
    'fig_e7_within_cluster_condition_split.png',
    'fig_e8_alluvial_cond_vs_path_a.png',
]
print('=== File verification ===')
all_ok = True
for fn in required_files:
    p = PHENO_PATH / fn
    ok = p.exists()
    print(f'  {fn}: {"OK" if ok else "MISSING"}')
    if not ok: all_ok = False
for fn in required_figs:
    p = FIG_PATH / fn
    ok = p.exists()
    print(f'  figures/{fn}: {"OK" if ok else "MISSING"}')
    if not ok: all_ok = False
print()
print('All files present:', all_ok)
print()
print('=== Assignment summary ===')
print(df['cond_cluster_full'].value_counts())
print()
print('=== Key ARI summary ===')
for c_name, ari in ari_c1.items():
    print(f'  ARI(within-cond {c_name} vs Path A): {ari:.4f}')
